In [ ]:
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

In [ ]:
from google.colab import drive
drive.mount('/content/drive')



In [ ]:
data = pd.read_csv('heart(1).csv')

In [ ]:
data.head()

In [ ]:
data.tail()

In [ ]:
data.info()

In [ ]:
data.describe()

In [ ]:
data

In [ ]:
df = data.copy()
df

In [ ]:
df = df.rename(columns={'Sex': 'Is_female'})
df

In [ ]:
# Inspect unique values and their types in the 'Is_female' column
print("Unique values in 'Is_female':", df['Is_female'].unique())
print("Value counts in 'Is_female':", df['Is_female'].value_counts(dropna=False))
print("Type of values in 'Is_female':", df['Is_female'].apply(type).value_counts())

The inspection will help us understand the exact values. Assuming the issue is whitespace or case, I'll modify the next cell to clean the values before mapping.

In [ ]:
df['ChestPainType'].value_counts()

In [ ]:
df.columns

In [ ]:
df.shape

In [ ]:
df.info()

In [ ]:
df.describe()

In [ ]:
df.duplicated().sum()

In [ ]:
df['HeartDisease'].value_counts().plot(kind = 'bar')

In [ ]:
df.isnull().sum()

In [ ]:
def plotting(var , num):
  plt.subplot(2,2,num)
  sns.histplot(df[var] , kde = True)

plotting('Age' , 1)
plotting('RestingBP' , 2)
plotting('Cholesterol' , 3)
plotting('MaxHR' , 4)
plt.tight_layout()
plt.show()


In [ ]:
df['Cholesterol'].value_counts()

In [ ]:
ch_mean = df.loc[df['Cholesterol'] != 0, 'Cholesterol'].mean()
ch_mean
# df['Cholesterol'] = df['Cholesterol'].replace(0, ch_mean)

In [ ]:
ch_mean = df.loc[df['Cholesterol'] != 0, 'Cholesterol'].mean()
df['Cholesterol'] = df['Cholesterol'].replace(0, ch_mean)
df

In [ ]:
df['Cholesterol'] = df['Cholesterol'].round(2)

In [ ]:
resting_bp_mean = df.loc[df['RestingBP'] != 0, 'RestingBP'].mean()
print(resting_bp_mean)
df['RestingBP'] = df['RestingBP'].replace(0, resting_bp_mean)
df['RestingBP'] = df['RestingBP'].round(2)
df['Cholesterol'].value_counts()
df['RestingBP'].value_counts()


In [ ]:
def plotting(var , num):
  plt.subplot(2,2,num)
  sns.histplot(df[var] , kde = True)

plotting('Age' , 1)
plotting('RestingBP' , 2)
plotting('Cholesterol' , 3)
plotting('MaxHR' , 4)
plt.tight_layout()
plt.show()


In [ ]:
sns.countplot(x = df['Is_female'])

In [ ]:
sns.countplot(x = df['Is_female'] , hue = df['HeartDisease'])

In [ ]:
sns.countplot(x = df['ChestPainType'])

In [ ]:
sns.countplot(x = df['ChestPainType'] , hue = df['HeartDisease'])

In [ ]:
sns.countplot(x = df['FastingBS'] , hue = df['HeartDisease'])

In [ ]:
sns.boxplot(x = 'HeartDisease' , y = 'Cholesterol'  , data = df)

In [ ]:
sns.violinplot(x = 'HeartDisease' , y = 'Age'  , data = df)

In [ ]:
sns.heatmap(df.corr(numeric_only= True) , annot = True)

In [ ]:
#Data PreProcessing and cleaning

In [ ]:
df_encode = pd.get_dummies(df , drop_first= False)

In [ ]:
df_encode

In [ ]:
df_encode = df_encode.astype(int)

In [ ]:
df_encode

In [ ]:
from sklearn.preprocessing import StandardScaler
df_encode.columns

In [ ]:
numerical_cols = ['Age', 'RestingBP', 'Cholesterol', 'FastingBS' , 'Oldpeak']
scaler = StandardScaler()
df_encode[numerical_cols] = scaler.fit_transform(df_encode[numerical_cols])
df_encode

In [ ]:
df_encode

In [ ]:
df_encode.columns

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.metrics import f1_score , accuracy_score , classification_report
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import GaussianNB
from sklearn.tree import DecisionTreeClassifier
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier

In [ ]:
x = df_encode.drop('HeartDisease' , axis = 1)
y = df_encode['HeartDisease']

In [ ]:
x_train , x_test , y_train , y_test = train_test_split(x , y , test_size = 0.2 , random_state = 42)

In [ ]:
x_train_scaled = scaler.fit_transform(x_train)
x_test_scaled = scaler.transform(x_test)

In [ ]:
models = {
    "Logistic Regression" : LogisticRegression(),
    "Naive Bayes" : GaussianNB(),
    "Decision Tree" : DecisionTreeClassifier(),
    "Support Vector Machine" : SVC(),
    "K-Nearest Neighbors" : KNeighborsClassifier()
}

In [ ]:
result = []

In [ ]:
for name , model in models.items():
    model.fit(x_train_scaled , y_train)
    y_pred = model.predict(x_test_scaled)
    accuracy = accuracy_score(y_test , y_pred)
    f1 = f1_score(y_test , y_pred)
    result.append({
        'Model' : name ,
        'Accuracy' : round(accuracy , 4) ,
        'F1-Score' : round(f1 , 4)
    })

In [ ]:
result

In [ ]:
import joblib
joblib.dump(models['K-Nearest Neighbors'] , 'KNN_heart.pkl')
joblib.dump(scaler , 'scaler.pkl')
joblib.dump(x.columns.tolist() , 'columns.pkl')

In [ ]:
x_train_scaled.columns